# 02 — SQL Avancé : Window Functions, CTEs, Subqueries

Ce notebook fait suite à `01_sql_basics.ipynb`. On monte d'un cran : les **window functions**, les **CTEs (Common Table Expressions)** et les **subqueries corrélées** — le niveau qui distingue un profil Data Science d'un simple utilisateur SQL.

**Contrairement au notebook précédent, on travaille ici sur des données réalistes** : un jeu de communes normandes et leurs relevés de pollution mensuels, cohérent avec le projet EcoSense — pas des tables jouets déconnectées.

## Sommaire
1. Setup — création de la base SQLite
2. Window Functions (ROW_NUMBER, RANK, DENSE_RANK, PARTITION BY, LAG, LEAD, moyenne mobile)
3. CTEs — WITH, simples et chaînées
4. Subqueries corrélées
5. Bonus — CTE récursive


## 1. Setup — création de la base SQLite

On construit une base à partir de deux tables :
- `communes` : les 6 villes principales du projet EcoSense (score JE, population, taux de pauvreté, région)
- `releves_pollution` : relevés mensuels de NO2 sur 12 mois pour chaque commune (avec variation saisonnière, comme dans le vrai projet)


In [1]:
import sqlite3
import pandas as pd
import numpy as np

np.random.seed(42)

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

def run_query(sql):
    """Exécute une requête SQL et retourne le résultat sous forme de DataFrame."""
    return pd.read_sql_query(sql, conn)


In [2]:
# --- Table communes ---
cur.execute('''
CREATE TABLE communes (
    commune_id INTEGER PRIMARY KEY,
    nom TEXT NOT NULL,
    region TEXT NOT NULL,
    population INTEGER,
    score_je REAL,
    taux_pauvrete REAL
)
''')

communes_data = [
    (1, "Le Havre",   "Seine-Maritime", 170000, 0.9041, 21.4),
    (2, "Rouen",      "Seine-Maritime", 110000, 0.5451, 15.2),
    (3, "Dieppe",     "Seine-Maritime",  30000, 0.5500, 16.7),
    (4, "Caen",       "Calvados",       105000, 0.1771,  9.8),
    (5, "Evreux",     "Eure",            48000, 0.4200, 13.1),
    (6, "Cherbourg",  "Manche",          78000, 0.3100, 11.0),
]

cur.executemany("INSERT INTO communes VALUES (?, ?, ?, ?, ?, ?)", communes_data)
conn.commit()

run_query("SELECT * FROM communes")


,commune_id,nom,region,population,score_je,taux_pauvrete
0,1,Le Havre,Seine-Maritime,170000,0.9041,21.4
1,2,Rouen,Seine-Maritime,110000,0.5451,15.2
2,3,Dieppe,Seine-Maritime,30000,0.5500,16.7
3,4,Caen,Calvados,105000,0.1771,9.8
4,5,Evreux,Eure,48000,0.4200,13.1
5,6,Cherbourg,Manche,78000,0.3100,11.0


In [3]:
# --- Table releves_pollution : 12 mois de relevés NO2 par commune ---
cur.execute('''
CREATE TABLE releves_pollution (
    releve_id INTEGER PRIMARY KEY AUTOINCREMENT,
    commune_id INTEGER NOT NULL,
    mois TEXT NOT NULL,
    no2 REAL,
    FOREIGN KEY (commune_id) REFERENCES communes(commune_id)
)
''')

# Valeurs de base cohérentes avec le projet EcoSense (no2_moyen annuel)
base_no2 = {1: 38.2, 2: 27.5, 3: 25.9, 4: 14.1, 5: 22.0, 6: 18.4}
mois_liste = [f"2025-{m:02d}" for m in range(1, 13)]

rows = []
for commune_id, base in base_no2.items():
    for i, mois in enumerate(mois_liste):
        # Saisonnalité : NO2 plus élevé en hiver (mois 1, 2, 11, 12)
        saison = 6 * np.cos(2 * np.pi * i / 12)
        bruit = np.random.normal(0, 1.5)
        valeur = round(base + saison + bruit, 1)
        rows.append((commune_id, mois, valeur))

cur.executemany("INSERT INTO releves_pollution (commune_id, mois, no2) VALUES (?, ?, ?)", rows)
conn.commit()

run_query("SELECT * FROM releves_pollution LIMIT 10")


,releve_id,commune_id,mois,no2
0,1,1,2025-01,44.9
1,2,1,2025-02,43.2
2,3,1,2025-03,42.2
3,4,1,2025-04,40.5
4,5,1,2025-05,34.8
5,6,1,2025-06,32.7
6,7,1,2025-07,34.6
7,8,1,2025-08,34.2
8,9,1,2025-09,34.5
9,10,1,2025-10,39.0


Base prête : **6 communes** et **72 relevés de pollution** (6 × 12 mois). Passons aux window functions.

## 2. Window Functions

Une window function calcule une valeur **pour chaque ligne**, en s'appuyant sur un groupe de lignes lié ("fenêtre" — d'où le nom), **sans réduire le nombre de lignes** contrairement à `GROUP BY`. C'est la différence fondamentale à retenir : `GROUP BY` agrège et compresse, une window function enrichit chaque ligne individuellement.

Syntaxe générale :
```sql
FONCTION() OVER (PARTITION BY colonne ORDER BY colonne)
```


### 2.1 — ROW_NUMBER() : classer les communes par score de justice environnementale

**Question métier :** quel est le classement des communes de la plus critique à la plus favorable ?

In [4]:
run_query('''
SELECT
    nom,
    region,
    score_je,
    ROW_NUMBER() OVER (ORDER BY score_je DESC) AS rang
FROM communes
''')


,nom,region,score_je,rang
0,Le Havre,Seine-Maritime,0.9041,1
1,Dieppe,Seine-Maritime,0.5500,2
2,Rouen,Seine-Maritime,0.5451,3
3,Evreux,Eure,0.4200,4
4,Cherbourg,Manche,0.3100,5
5,Caen,Calvados,0.1771,6


### 2.2 — RANK() vs DENSE_RANK() : gérer les ex-aequo

`ROW_NUMBER()` attribue toujours un rang différent, même en cas d'égalité — ce n'est pas toujours ce qu'on veut. `RANK()` laisse un "trou" après une égalité (2 communes en rang 1 → la suivante est rang 3). `DENSE_RANK()` ne laisse pas de trou (2 communes en rang 1 → la suivante est rang 2).

On illustre avec le taux de pauvreté arrondi à l'entier, qui crée artificiellement des égalités.

In [5]:
run_query('''
SELECT
    nom,
    taux_pauvrete,
    ROUND(taux_pauvrete) AS taux_arrondi,
    RANK() OVER (ORDER BY ROUND(taux_pauvrete) DESC) AS rang_avec_trous,
    DENSE_RANK() OVER (ORDER BY ROUND(taux_pauvrete) DESC) AS rang_sans_trous
FROM communes
''')


,nom,taux_pauvrete,taux_arrondi,rang_avec_trous,rang_sans_trous
0,Le Havre,21.4,21.0,1,1
1,Dieppe,16.7,17.0,2,2
2,Rouen,15.2,15.0,3,3
3,Evreux,13.1,13.0,4,4
4,Cherbourg,11.0,11.0,5,5
5,Caen,9.8,10.0,6,6


### 2.3 — PARTITION BY : le relevé de pollution le plus élevé, par commune

**Question métier :** quel est le pic de pollution NO2 enregistré pour chaque commune sur l'année ?

`PARTITION BY` redémarre le calcul de la window function à zéro pour chaque groupe — ici, un classement des relevés séparé pour chaque commune.

In [6]:
run_query('''
SELECT nom, mois, no2, rang
FROM (
    SELECT
        c.nom,
        r.mois,
        r.no2,
        ROW_NUMBER() OVER (PARTITION BY r.commune_id ORDER BY r.no2 DESC) AS rang
    FROM releves_pollution r
    JOIN communes c ON c.commune_id = r.commune_id
)
WHERE rang = 1
ORDER BY no2 DESC
''')


,nom,mois,no2,rang
0,Le Havre,2025-01,44.9,1
1,Rouen,2025-01,33.9,1
2,Dieppe,2025-02,31.3,1
3,Evreux,2025-12,28.7,1
4,Cherbourg,2025-12,25.9,1
5,Caen,2025-12,20.9,1


### 2.4 — LAG() : variation mensuelle du NO2

**Question métier :** de combien la pollution a-t-elle varié par rapport au mois précédent, pour chaque commune ?

`LAG()` va chercher la valeur de la ligne précédente (dans l'ordre défini par `ORDER BY`, à l'intérieur de chaque `PARTITION BY`). `LEAD()` fait l'inverse : il va chercher la ligne suivante.

In [7]:
run_query('''
SELECT
    c.nom,
    r.mois,
    r.no2,
    LAG(r.no2) OVER (PARTITION BY r.commune_id ORDER BY r.mois) AS no2_mois_precedent,
    ROUND(r.no2 - LAG(r.no2) OVER (PARTITION BY r.commune_id ORDER BY r.mois), 1) AS variation
FROM releves_pollution r
JOIN communes c ON c.commune_id = r.commune_id
WHERE c.nom = 'Le Havre'
ORDER BY r.mois
''')


,nom,mois,no2,no2_mois_precedent,variation
0,Le Havre,2025-01,44.9,NaN,NaN
1,Le Havre,2025-02,43.2,44.9,-1.7
2,Le Havre,2025-03,42.2,43.2,-1.0
3,Le Havre,2025-04,40.5,42.2,-1.7
4,Le Havre,2025-05,34.8,40.5,-5.7
5,Le Havre,2025-06,32.7,34.8,-2.1
6,Le Havre,2025-07,34.6,32.7,1.9
7,Le Havre,2025-08,34.2,34.6,-0.4
8,Le Havre,2025-09,34.5,34.2,0.3
9,Le Havre,2025-10,39.0,34.5,4.5


### 2.5 — Moyenne mobile sur 3 mois glissants

**Question métier :** quelle est la tendance de fond du NO2, en lissant les variations mensuelles ponctuelles ?

`ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` définit une fenêtre glissante de 3 lignes (la ligne courante + les 2 précédentes).

In [8]:
run_query('''
SELECT
    c.nom,
    r.mois,
    r.no2,
    ROUND(AVG(r.no2) OVER (
        PARTITION BY r.commune_id
        ORDER BY r.mois
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 1) AS moyenne_mobile_3mois
FROM releves_pollution r
JOIN communes c ON c.commune_id = r.commune_id
WHERE c.nom = 'Le Havre'
ORDER BY r.mois
''')


,nom,mois,no2,moyenne_mobile_3mois
0,Le Havre,2025-01,44.9,44.9
1,Le Havre,2025-02,43.2,44.0
2,Le Havre,2025-03,42.2,43.4
3,Le Havre,2025-04,40.5,42.0
4,Le Havre,2025-05,34.8,39.2
5,Le Havre,2025-06,32.7,36.0
6,Le Havre,2025-07,34.6,34.0
7,Le Havre,2025-08,34.2,33.8
8,Le Havre,2025-09,34.5,34.4
9,Le Havre,2025-10,39.0,35.9


> 💡 **À retenir :** la moyenne mobile est LA technique de base en analyse de séries temporelles pour lisser le bruit et révéler une tendance — tu la retrouveras autant en SQL qu'en pandas (`.rolling()`) qu'en amont d'un modèle SARIMA (comme dans EcoSense).

## 3. CTEs (Common Table Expressions) — `WITH`

Une CTE nomme un résultat intermédiaire, utilisable dans le reste de la requête comme une table temporaire. Deux bénéfices concrets : la requête devient **lisible** (on découpe un problème complexe en étapes nommées), et on peut **réutiliser** le même résultat intermédiaire plusieurs fois sans le recalculer.

### 3.1 — CTE simple : communes plus polluées que la moyenne régionale

**Question métier :** quelles communes dépassent la moyenne de NO2 de leur propre région ?

In [9]:
run_query('''
WITH moyenne_regionale AS (
    SELECT
        c.region,
        AVG(r.no2) AS no2_moyen_region
    FROM releves_pollution r
    JOIN communes c ON c.commune_id = r.commune_id
    GROUP BY c.region
)
SELECT
    c.nom,
    c.region,
    ROUND(AVG(r.no2), 1) AS no2_moyen_commune,
    ROUND(m.no2_moyen_region, 1) AS no2_moyen_region
FROM releves_pollution r
JOIN communes c ON c.commune_id = r.commune_id
JOIN moyenne_regionale m ON m.region = c.region
GROUP BY c.nom
HAVING AVG(r.no2) > m.no2_moyen_region
''')


,nom,region,no2_moyen_commune,no2_moyen_region
0,Le Havre,Seine-Maritime,38.6,30.3


### 3.2 — CTEs chaînées : pipeline en plusieurs étapes

**Question métier :** quelles communes cumulent un score JE élevé (top 50%) ET un NO2 moyen annuel supérieur à 20 µg/m³ ?

On enchaîne deux CTEs : la première calcule le NO2 moyen annuel par commune, la seconde s'appuie sur la première pour filtrer.

In [10]:
run_query('''
WITH no2_annuel AS (
    SELECT
        commune_id,
        AVG(no2) AS no2_moyen_annuel
    FROM releves_pollution
    GROUP BY commune_id
),
communes_enrichies AS (
    SELECT
        c.nom,
        c.score_je,
        n.no2_moyen_annuel,
        CASE WHEN c.score_je >= (SELECT AVG(score_je) FROM communes) THEN 'Au-dessus médiane' ELSE 'En-dessous médiane' END AS position_score_je
    FROM communes c
    JOIN no2_annuel n ON n.commune_id = c.commune_id
)
SELECT *
FROM communes_enrichies
WHERE position_score_je = 'Au-dessus médiane' AND no2_moyen_annuel > 20
ORDER BY no2_moyen_annuel DESC
''')


,nom,score_je,no2_moyen_annuel,position_score_je
0,Le Havre,0.9041,38.650,Au-dessus médiane
1,Rouen,0.5451,26.625,Au-dessus médiane
2,Dieppe,0.5500,25.625,Au-dessus médiane


## 4. Subqueries corrélées

Une subquery corrélée **référence une colonne de la requête externe** — elle est donc réévaluée pour chaque ligne de la requête externe, contrairement à une subquery classique calculée une seule fois. C'est plus coûteux en performance mais parfois la façon la plus naturelle d'exprimer une comparaison "par rapport à son propre groupe".

### 4.1 — Communes dont le score JE dépasse la moyenne de leur région

Même résultat conceptuel que l'exercice 3.1, mais écrit avec une subquery corrélée plutôt qu'une CTE — pour bien voir la différence de style.

In [11]:
run_query('''
SELECT
    c1.nom,
    c1.region,
    c1.score_je,
    (SELECT ROUND(AVG(c2.score_je), 3)
     FROM communes c2
     WHERE c2.region = c1.region) AS score_je_moyen_region
FROM communes c1
WHERE c1.score_je > (
    SELECT AVG(c2.score_je)
    FROM communes c2
    WHERE c2.region = c1.region   -- ⚡ corrélation : dépend de la ligne externe c1
)
''')


,nom,region,score_je,score_je_moyen_region
0,Le Havre,Seine-Maritime,0.9041,0.666


> ⚠️ **Piège classique  :** si tu retires la clause `WHERE c2.region = c1.region` dans la subquery, elle n'est plus corrélée — elle compare alors chaque commune à la moyenne **globale** (toutes régions confondues) plutôt qu'à la moyenne de sa propre région. Le résultat change complètement. Sache toujours pointer précisément ce qui rend une subquery "corrélée" ou non.

## 5. Bonus — CTE récursive

Une CTE récursive s'appelle elle-même jusqu'à ce qu'une condition d'arrêt soit atteinte. Usage classique : générer une séquence, ou parcourir une hiérarchie (organigramme, arborescence de catégories).

**Exemple simple :** générer la liste des 12 mois de 2025 sans les stocker en dur — utile si on doit détecter des mois sans relevé (mois manquants dans `releves_pollution`).

In [12]:
run_query('''
WITH RECURSIVE mois_generes(n, mois) AS (
    SELECT 1, '2025-01'
    UNION ALL
    SELECT n + 1, '2025-' || substr('0' || (n + 1), -2, 2)
    FROM mois_generes
    WHERE n < 12
)
SELECT mois FROM mois_generes
''')


,mois
0,2025-01
1,2025-02
2,2025-03
3,2025-04
4,2025-05
5,2025-06
6,2025-07
7,2025-08
8,2025-09
9,2025-10


On peut maintenant s'en servir pour repérer d'éventuels mois manquants dans les relevés d'une commune — un contrôle qualité très utile sur des données réelles (souvent incomplètes, contrairement à notre jeu synthétique ici) :

In [13]:
run_query('''
WITH RECURSIVE mois_generes(n, mois) AS (
    SELECT 1, '2025-01'
    UNION ALL
    SELECT n + 1, '2025-' || substr('0' || (n + 1), -2, 2)
    FROM mois_generes
    WHERE n < 12
)
SELECT m.mois
FROM mois_generes m
LEFT JOIN releves_pollution r
    ON r.mois = m.mois AND r.commune_id = 1  -- Le Havre
WHERE r.releve_id IS NULL
''')


,mois


*(Résultat vide attendu ici puisque notre jeu de données synthétique est complet sur les 12 mois — mais la requête ci-dessus est le genre d'outil de contrôle qualité à avoir en tête sur des données réelles, typiquement incomplètes.)*

## Récapitulatif

| Concept | À retenir |
|---|---|
| `ROW_NUMBER()` | Rang unique, même en cas d'égalité |
| `RANK()` | Laisse des trous après une égalité |
| `DENSE_RANK()` | Pas de trous après une égalité |
| `PARTITION BY` | Redémarre le calcul pour chaque groupe (sans réduire les lignes, contrairement à GROUP BY) |
| `LAG()` / `LEAD()` | Valeur de la ligne précédente / suivante |
| `ROWS BETWEEN ... AND ...` | Définit une fenêtre glissante (ex : moyenne mobile) |
| CTE (`WITH`) | Nomme un résultat intermédiaire, lisible et réutilisable |
| CTEs chaînées | Pipeline de transformations en plusieurs étapes nommées |
| Subquery corrélée | Référence une colonne de la requête externe, réévaluée ligne par ligne |
| CTE récursive | S'appelle elle-même — génération de séquences ou parcours de hiérarchies |

## Prochaines étapes

- ✅ `01_sql_basics.ipynb` — terminé
- ✅ `02_sql_advanced.ipynb` — terminé (ce notebook)
- 📅 `03_pandas_advanced.ipynb` — reprendre chaque requête SQL ci-dessus et écrire son équivalent pandas (`groupby`, `rank()`, `shift()` pour LAG/LEAD, `rolling()` pour la moyenne mobile) — pour pouvoir répondre à *"tu ferais ça en SQL ou en pandas ?"* en connaissance de cause
- 📅 Volet Oracle — recréer une sélection de ces requêtes sur une vraie base Oracle via DataGrip, en notant les différences de syntaxe
